#Notebook: 02_silver_tratamento.sql

Este notebook transforma dados da camada bronze (raw/string-first) para a camada silver aplicando: tipagem, deduplicação, regras de qualidade, rejeição com motivo e persistência idempotente via MERGE (DELTA).

##Objetivos:

- Entregar tabelas "cleansed & validated" para consumo analítico e para a camada gold
- Garantir rastreabilidade (ingestion_ts, load_id, soucr_file)
- Controlar incrementalidade por ingestion_ts com checkpoint

##Estratégia:

- SQL-first (TEMPO VIEWs para staging)
- TRY_CAST/TRY_TO_TIMESTAMPO para tipagem resiliente
- row_number() para dedupe determinístico
- rejets em tabelas dedicadas com reject_reason
- MERGE idempotente com row_hash
- checkpoint por tabela para incremental



##Contexto

In [0]:
%sql
USE CATALOG healthcare_dev

##Cria schemas

In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS healthcare_dev.silver;
CREATE SCHEMA IF NOT EXISTS healthcare_dev.silver_rejects;
CREATE SCHEMA IF NOT EXISTS healthcare_dev.silver_ops;

##Checkpoint para incrementalidade por tabela

In [0]:
%sql
CREATE TABLE IF NOT EXISTS healthcare_dev.silver_ops.pipeline_checkpoint (
  table_name STRING,
  last_ingestion_ts TIMESTAMP,
  last_run_ts TIMESTAMP,
  status STRING, 
  rows_read BIGINT,
  rows_written BIGINT,
  rows_rejected BIGINT
)
USING DELTA;

##silver.fact_contratos 

Tratamento da bronze.contratos_planos

In [0]:
%sql
CREATE TABLE IF NOT EXISTS healthcare_dev.silver.fact_contratos (
  contrato_id STRING,
  beneficiario_id STRING,
  tipo_plano STRING,
  valor_mensal DECIMAL(10,2),
  coparticipacao_flag STRING,
  data_inicio_vigencia TIMESTAMP,
  data_fim_vigencia TIMESTAMP,
  ingestion_ts TIMESTAMP,
  load_id STRING,
  source_file STRING,
  row_hash STRING
)
USING DELTA;

##SILVER: rejects.fact_contratos

In [0]:
%sql
CREATE TABLE IF NOT EXISTS healthcare_dev.silver_rejects.fact_contratos (
  contrato_id STRING,
  beneficiario_id STRING,
  tipo_plano STRING,
  valor_mensal STRING,
  coparticipacao_flag STRING,
  data_inicio_vigencia STRING,
  data_fim_vigencia STRING,
  ingestion_ts TIMESTAMP,
  load_id STRING,
  source_file STRING,
  reject_reason STRING,
  reject_ts TIMESTAMP
)
USING DELTA;

##Staging: incremental por ingestion_ts

In [0]:
%sql
CREATE OR REPLACE TEMP VIEW stage_contratos_raw AS
WITH last_cp AS (
  SELECT coalesce(max(last_ingestion_ts), timestamp('1900-01-01')) AS last_ingestion_ts
  FROM healthcare_dev.silver_ops.pipeline_checkpoint
  WHERE table_name = 'fact_contratos'
)
SELECT *
FROM healthcare_dev.bronze.contratos_planos
WHERE ingestion_ts >= (SELECT last_ingestion_ts FROM last_cp) - INTERVAL 3 DAYS;

##Tipagem / Padronização

In [0]:
%sql
CREATE OR REPLACE TEMP VIEW stg_contratos_typed AS
SELECT
  cast(contrato_id as string) AS contrato_id,
  cast(beneficiario_id as string) AS beneficiario_id,
  upper(trim(cast(tipo_plano as string))) AS tipo_plano,
  try_cast(valor_mensal as decimal(10,2)) AS valor_mensal,
  upper(trim(cast(coparticipacao_flag as string))) AS coparticipacao_flag,
  try_to_timestamp(data_inicio_vigencia) AS data_inicio_vigencia,
  try_to_timestamp(data_fim_vigencia) AS data_fim_vigencia,
  ingestion_ts,
  load_id,
  source_file
FROM stg_contratos_raw;


##Deduplicação determinística (contrato_id) mantendo o mais recente por ingestion_ts

In [0]:
%sql
CREATE OR REPLACE TEMP VIEW stg_contratos_dedup AS
SELECT *
FROM (
  SELECT
    *,
    row_number() OVER(
      PARTITION BY contrato_id
      ORDER BY ingestion_ts DESC 
    ) AS rn
  FROM stg_contratos_typed
) x
WHERE rn = 1;

##Classificação: válido vs reject (com motivo)

In [0]:
%sql
CREATE OR REPLACE TEMP VIEW stg_contratos_classified AS
SELECT
  *,
  CASE 
    WHEN contrato_id IS NULL OR contrato_id = '' THEN 'MISSING_CONTRATO_ID'
    WHEN beneficiario_id IS NULL OR beneficiario_id = '' THEN 'MISSING_BENEFICIARIO_ID'
    WHEN data_inicio_vigencia IS NULL THEN 'INVALID_DATA_INICIO_VIGENCIA'
    WHEN data_fim_vigencia IS NOT NULL AND data_fim_vigencia < data_inicio_vigencia THEN 'DATA_FIM_LT_INICIO' 
    WHEN valor_mensal IS NOT NULL AND valor_mensal < 0 THEN 'NEGATIVE_VALOR_MENSAL'
    ELSE NULL 
  END AS reject_reason
FROM stg_contratos_dedup;

##Persistir rejects

In [0]:
%sql
INSERT INTO healthcare_dev.silver_rejects.fact_contratos
SELECT
  contrato_id,
  beneficiario_id,
  tipo_plano,
  cast(valor_mensal as string) AS valor_mensal,
  coparticipacao_flag,
  cast(data_inicio_vigencia as string) AS data_inicio_vigencia,
  cast(data_fim_vigencia as string) AS data_fim_vigencia,
  ingestion_ts,
  load_id,
  source_file,
  reject_reason,
  current_timestamp() AS reject_ts
FROM stg_contratos_classified
WHERE reject_reason IS NOT NULL;

##Valid + row_hash

In [0]:
%sql
CREATE OR REPLACE TEMP VIEW stg_contratos_valid AS
SELECT
  contrato_id,
  beneficiario_id,
  tipo_plano,
  valor_mensal,
  coparticipacao_flag,
  data_inicio_vigencia,
  data_fim_vigencia,
  ingestion_ts,
  load_id,
  source_file,
  sha2(concat_ws('||',
    contrato_id,
    coalesce(beneficiario_id,''),
    coalesce(tipo_plano,''),
    coalesce(cast(valor_mensal as string),''),
    coalesce(coparticipacao_flag,''),
    coalesce(cast(data_inicio_vigencia as string),''),
    coalesce(cast(data_fim_vigencia as string),'')
  ), 256) AS row_hash
FROM stg_contratos_classified
WHERE reject_reason IS NULL;

##MERGE idempotente

In [0]:
%sql
MERGE INTO healthcare_dev.silver.fatc_contratos AS tgt
USING stg_contratos_valid AS src
ON tgt.contrato_id = src.conotrato_id
WHEN MATCHED AND tgt.row_hash <> drc.row_hash THEN UPDATE SET *
WHEN NOT MATCHED THEN INSERT *;

##Atualizar checkpoint

In [0]:
%sql
INSERT INTO healthcare_dev.silver_ops.pipeline_checkpoint
SELECT
  'fact_contratos' AS table_name,
  max(ingestion_ts) AS last_ingestion_ts,
  current_timestamp() AS last_run_ts,
  'SUCESS' AS status,
  null AS rows_read,
  null AS rows_written,
  null AS rows_rejected
FROM stg_contratos_valid; 

##SILVER: fact_faturas

In [0]:
%sql
CREATE TABLE IF NOT EXISTS healthcare_dev.silver.fact_faturas (
  beneficiario_id STRING,
  competencia STRING,
  data_pagamento DATE,
  status_pagamento STRING,
  valor_faturado DECIMAL(18,2),
  valor_pago DECIMAL(18,2),
  ingestion_ts TIMESTAMP,
  load_id STRING,
  source_file STRING,
  row_hash STRING
)
USING DELTA;

##silver_rejects.fact_faturas

In [0]:
%sql
CREATE TABLE IF NOT EXISTS healthcare_dev.silver_rejects.fact_faturas (
  beneficiario_id STRING,
  competencia STRING,
  data_pagamento STRING,
  status_pagamento STRING,
  valor_faturado STRING,
  valor_pago STRING,
  ingestion_ts TIMESTAMP,
  load_id STRING,
  source_file STRING,
  reject_reason STRING,
  reject_ts TIMESTAMP
)
USING DELTA;

##Staging incremental

In [0]:
%sql
CREATE OR REPLACE TEMP VIEW stg_faturas_raw AS
WITH last_cp AS (
  SELECT coalesce(max(last_ingestion_ts), timestamp('1900-01-01')) AS last_ingestion_ts
  FROM healthcare_dev.silver_ops.pipeline_checkpoint
  WHERE table_name = 'fact_faturas'
)
SELECT *
FROM healthcare_dev.bronze.faturas_pagamentos
WHERE ingestion_ts >= (SELECT last_ingestion_ts FROM last_cp) - INTERVAL 3 DAYS

##Tipagem

In [0]:
%sql
CREATE OR REPLACE TEMP VIEW stg_faturas_typed AS
SELECT
  cast(beneficiario_id as string) AS beneficiario_id,
  upper(trim(cast(competencia as string))) AS competencia,
  try_cast(data_pagamento as date) AS data_pagamento,
  upper(trim(cast(status_pagamento as string))) AS status_pagamento,
  try_cast(valor_faturado as decimal(18,2)) AS valor_faturado,
  try_cast(valor_pago as decimal(18,2)) AS valor_pago,
  ingestion_ts,
  load_id,
  source_file
FROM stg_faturas_raw;

##Dedup (beneficiario_id, competencia)

In [0]:
%sql
CREATE OR REPLACE TEMP VIEW stg_faturas_dedup AS
SELECT *
FROM (
  SELECT
    *,
    row_number() OVER (
      PARTITION BY beneficiario_id, competencia
      ORDER BY ingestion_ts DESC
    ) AS rn
  FROM stg_faturas_typed
) x
WHERE rn = 1;

##Classificação

In [0]:
%sql
CREATE OR REPLACE TEMP VIEW stg_faturas_classified AS
SELECT
  *,
  CASE
    WHEN beneficiario_id IS NULL OR beneficiario_id = '' THEN 'MISSING_BENEFICIARIO_ID'
    WHEN competencia IS NULL OR competencia = '' THEN 'MISSING_COMPETENCIA'
    WHEN valor_faturado IS NULL THEN 'INVALID_VALOR_FATURADO'
    WHEN valor_faturado < 0 THEN 'NEGATIVE_VALOR_FATURADO'
    WHEN valor_pago IS NOT NULL AND valor_pago < 0 THEN 'NEGATIVE_VALOR_PAGO'
    WHEN valor_pago IS NOT NULL AND valor_pago > valor_faturado THEN 'VALOR_PAGO_GT_FATURADO'
    ELSE NULL
  END AS reject_reason
FROM stg_faturas_dedup;

##Persistir rejects

In [0]:
%sql
INSERT INTO healthcare_dev.silver_rejects.fact_faturas
SELECT
  beneficiario_id,
  competencia,
  cast(data_pagamento as string) AS data_pagamento,
  status_pagamento,
  cast(valor_faturado as string) AS valor_faturado,
  cast(valor_pago as string) AS valor_pago,
  ingestion_ts,
  load_id,
  source_file,
  reject_reason,
  current_timestamp() AS reject_ts
FROM stg_faturas_classified
WHERE reject_reason IS NOT NULL;

##Valid + row_hash

In [0]:
%sql
CREATE OR REPLACE TEMP VIEW stg_faturas_valid AS
SELECT
  beneficiario_id,
  competencia,
  data_pagamento,
  status_pagamento,
  valor_faturado,
  valor_pago,
  ingestion_ts,
  load_id,
  source_file,
  sha2(concat_ws('||',
    beneficiario_id,
    competencia,
    coalesce(cast(data_pagamento as string),''),
    coalesce(status_pagamento,''),
    coalesce(cast(valor_faturado as string),''),
    coalesce(cast(valor_pago as string),'')
  ), 256) AS row_hash
FROM stg_faturas_classified
WHERE reject_reason IS NULL;

##MERGE

In [0]:
%sql
MERGE INTO healthcare_dev.silver.fact_faturas AS tgt
USING stg_faturas_valid AS src
ON tgt.beneficiario_id = src.beneficiario_id
AND tgt.competencia = src.competencia
WHEN MATCHED AND tgt.row_hash <> src.row_hash THEN UPDATE SET *
WHEN NOT MATCHED THEN INSERT *;

##Checkpoint

In [0]:
%sql
INSERT INTO healthcare_dev.silver_ops.pipeline_checkpoint
SELECT
  'fact_faturas' AS table_name,
  max(ingestion_ts) AS last_ingestion_ts,
  current_timestamp() AS last_run_ts,
  'SUCCESS' AS status,
  null AS rows_read,
  null AS rows_written,
  null AS rows_rejected
FROM stg_faturas_valid;

##SILVER: fact_logs

In [0]:
%sql
CREATE TABLE IF NOT EXISTS healthcare_dev.silver.fact_app_logs (
  event_id STRING,
  beneficiario_id STRING,
  canal STRING,
  evento STRING,
  http_status INT,
  latencia_ms INT,
  sessao_id STRING,
  timestamp_evento TIMESTAMP,
  ingestion_ts TIMESTAMP,
  load_id STRING,
  source_file STRING,
  row_hash STRING
)
USING DELTA;

##silver_rejects.fact_app_logs

In [0]:
%sql
CREATE TABLE IF NOT EXISTS healthcare_dev.silver_rejects.fact_app_logs (
  event_id STRING,
  beneficiario_id STRING,
  canal STRING,
  evento STRING,
  http_status STRING,
  latencia_ms STRING,
  sessao_id STRING,
  timestamp_evento STRING,
  ingestion_ts TIMESTAMP,
  load_id STRING,
  source_file STRING,
  reject_reason STRING,
  reject_ts TIMESTAMP
)
USING DELTA;

##Raw incremental

In [0]:
%sql
CREATE OR REPLACE TEMP VIEW stg_logs_raw AS
WITH last_cp AS (
  SELECT coalesce(max(last_ingestion_ts), timestamp('1900-01-01')) AS last_ingestion_ts
  FROM healthcare_dev.silver_ops.pipeline_checkpoint
  WHERE table_name = 'fact_app_logs'
)
SELECT *
FROM healthcare_dev.bronze.app_event_log
WHERE ingestion_ts >= (SELECT last_ingestion_ts FROM last_cp) - INTERVAL 3 DAYS;

##Tipagem

In [0]:
%sql
CREATE OR REPLACE TEMP VIEW stg_logs_typed AS
SELECT
  cast(event_id as string) AS event_id,
  cast(beneficiario_id as string) AS beneficiario_id,
  upper(trim(cast(canal as string))) AS canal,
  upper(trim(cast(evento as string))) AS evento,
  try_cast(http_status as int) AS http_status,
  try_cast(latencia_ms as int) AS latencia_ms,
  cast(sessao_id as string) AS sessao_id,
  try_to_timestamp(timestamp_evento) AS timestamp_evento,
  ingestion_ts,
  load_id,
  source_file
FROM stg_logs_raw;

##Dedup por event_id

In [0]:
%sql
CREATE OR REPLACE TEMP VIEW stg_logs_dedup AS
SELECT *
FROM (
  SELECT
    *,
    row_number() OVER (
      PARTITION BY event_id
      ORDER BY ingestion_ts DESC
    ) AS rn
  FROM stg_logs_typed
) x
WHERE rn = 1;

##Clissificação

In [0]:
%sql
CREATE OR REPLACE TEMP VIEW stg_logs_classified AS
SELECT
  *,
  CASE
    WHEN event_id IS NULL OR event_id = '' THEN 'MISSING_EVENT_ID'
    WHEN timestamp_evento IS NULL THEN 'INVALID_TIMESTAMP_EVENTO'
    WHEN http_status IS NOT NULL AND (http_status < 100 OR http_status > 599) THEN 'INVALID_HTTP_STATUS'
    WHEN latencia_ms IS NOT NULL AND latencia_ms < 0 THEN 'NEGATIVE_LATENCIA_MS'
    ELSE NULL
  END AS reject_reason
FROM stg_logs_dedup;

##Rejects

In [0]:
%sql
INSERT INTO healthcare_dev.silver_rejects.fact_app_logs
SELECT
  event_id,
  beneficiario_id,
  canal,
  evento,
  cast(http_status as string) AS http_status,
  cast(latencia_ms as string) AS latencia_ms,
  sessao_id,
  cast(timestamp_evento as string) AS timestamp_evento,
  ingestion_ts,
  load_id,
  source_file,
  reject_reason,
  current_timestamp() AS reject_ts
FROM stg_logs_classified
WHERE reject_reason IS NOT NULL;

##Valid = hash

In [0]:
%sql
CREATE OR REPLACE TEMP VIEW stg_logs_valid AS
SELECT
  event_id,
  beneficiario_id,
  canal,
  evento,
  http_status,
  latencia_ms,
  sessao_id,
  timestamp_evento,
  ingestion_ts,
  load_id,
  source_file,
  sha2(concat_ws('||',
    event_id,
    coalesce(beneficiario_id,''),
    coalesce(canal,''),
    coalesce(evento,''),
    coalesce(cast(http_status as string),''),
    coalesce(cast(latencia_ms as string),''),
    coalesce(sessao_id,''),
    coalesce(cast(timestamp_evento as string),'')
  ), 256) AS row_hash
FROM stg_logs_classified
WHERE reject_reason IS NULL;

##MERGE

In [0]:
%sql
MERGE INTO healthcare_dev.silver.fact_app_logs AS tgt
USING stg_logs_valid AS src
ON tgt.event_id = src.event_id
WHEN MATCHED AND tgt.row_hash <> src.row_hash THEN UPDATE SET *
WHEN NOT MATCHED THEN INSERT *;

##Checkpoint

In [0]:
%sql
INSERT INTO healthcare_dev.silver_ops.pipeline_checkpoint
SELECT
  'fact_app_logs' AS table_name,
  max(ingestion_ts) AS last_ingestion_ts,
  current_timestamp() AS last_run_ts,
  'SUCCESS' AS status,
  null AS rows_read,
  null AS rows_written,
  null AS rows_rejected
FROM stg_logs_valid;

##Valida pós carga

In [0]:
%sql
SELECT
  (SELECT COUNT(*) FROM healthcare_dev.silver.fact_contratos) AS n_contratos_silver,
  (SELECT COUNT(*) FROM healthcare_dev.silver.fact_faturas)   AS n_faturas_silver,
  (SELECT COUNT(*) FROM healthcare_dev.silver.fact_app_logs)  AS n_logs_silver;

%sql
SELECT
  (SELECT COUNT(*) FROM healthcare_dev.silver_rejects.fact_contratos) AS n_contratos_rejects,
  (SELECT COUNT(*) FROM healthcare_dev.silver_rejects.fact_faturas)   AS n_faturas_rejects,
  (SELECT COUNT(*) FROM healthcare_dev.silver_rejects.fact_app_logs)  AS n_logs_rejects;

In [0]:
%sql
-- ATENÇÃO: Isso apaga tudo dentro desses schemas
DROP SCHEMA IF EXISTS healthcare_dev.silver CASCADE;
DROP SCHEMA IF EXISTS healthcare_dev.silver_rejects CASCADE;
DROP SCHEMA IF EXISTS healthcare_dev.silver_ops CASCADE;

In [0]:
%sql
SHOW TABLES IN healthcare_dev.silver;
SHOW TABLES IN healthcare_dev.silver_rejects;
SHOW TABLES IN healthcare_dev.silver_ops;